<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2010%20-%202026-05-06%20-%20Visualizaciones%20avanzadas/clase_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 10 · Visualizaciones avanzadas

En los últimos días vimos cómo explorar datos: stats básicas, correlaciones, distribuciones. Todo bien. Pero un matplotlib simple solo te lleva hasta ahí. Hoy vamos a ver cómo hacer visualizaciones que realmente sirven para contar historias.

Seaborn es tu amiga porque piensa como un analista de datos. Plotly es útil cuando necesitas que la gente interactúe con los gráficos. Y saber cuándo usar cada una es casi más importante que saber escribir el código.

(Ejecuta cada celda con `Shift + Enter`.)

> **Hoy haces** · Los tres ejercicios con Seaborn y Plotly, las dos pausas y el checkpoint. ~2h.
>
> **Entrega** · El notebook ejecutado al repo del equipo antes de la próxima clase.

## Setup

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

print("Setup completo ✓")
print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")

---

## El problema

Imagina que tienes el Titanic: miles de datos sobre pasajeros, clases, edades, tarifas. Una table de pandas te dice que tal variable correlaciona con tal otra. Pero ¿cuál es el patrón visual? ¿Las mujeres de primera clase sobrevivieron más? ¿La edad importa realmente?

Hoy vamos a ver cuatro herramientas: **pairplot** (ve muchas variables de golpe), **FacetGrid** (compara el mismo gráfico entre grupos), **heatmap de correlación** (qué se relaciona con qué), y **Plotly** (cuando alguien necesita clickear e interactuar).

In [ ]:
titanic = sns.load_dataset("titanic")

print(f"Shape: {titanic.shape}")
print(f"\nNulls:\n{titanic.isnull().sum()}")
print(f"\nColumnas: {titanic.columns.tolist()}")
titanic.head()

¿Ves aquí 11 columnas y casi 900 filas? Es demasiado para entender de una vez en una tabla. Vamos a hacer que los ojos lo entienda.

---

## 1. Pairplot · la matriz de todo contra todo

Pairplot crea una matriz donde cada par de variables numéricas aparece como un scatter plot. La diagonal muestra la distribución de cada variable sola (histograma o densidad). Parece complicado pero es increíblemente útil para ver patrones rápido.

In [ ]:
vars_numericas = ["age", "fare", "survived"]
df_clean = titanic[vars_numericas].dropna()

g = sns.pairplot(
    df_clean,
    hue="survived",
    diag_kind="hist",
    plot_kws={"alpha": 0.6},
    height=2.5
)
g.fig.suptitle("Pairplot: Edad, Tarifa y Supervivencia", y=1.01, fontsize=12)
plt.tight_layout()

Rojo = murió, azul = sobrevivió. Mirá cómo los colores se agrupan en lugares distintos en los gráficos. Eso es lo que buscas: un patrón visual claro.

### Ejercicio 1

Crea un pairplot del dataset `penguins` con `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` coloreado por especie. Usa `diag_kind="kde"` (densidad, no histograma — suena más profesional).

In [ ]:
penguins = sns.load_dataset("penguins")

columnas_interes = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "species"]
df_peng = penguins[columnas_interes].dropna()

pp = sns.pairplot(
    df_peng,
    hue="___",
    diag_kind="___",
    height=2
)
pp.fig.suptitle("Pairplot: Medidas de pingüinos", y=1.01)
plt.tight_layout()

In [ ]:
assert "pp" in dir(), "La variable pp no existe"
assert pp.fig is not None, "El pairplot no se creó correctamente"
print("✅ Ejercicio 1 completado")

In [ ]:
# %% [solution]
# pp = sns.pairplot(
#     df_peng,
#     hue="species",
#     diag_kind="kde",
#     height=2
# )
# pp.fig.suptitle("Pairplot: Medidas de pingüinos", y=1.01)
# plt.tight_layout()

---

## 2. FacetGrid · divide y compara

A veces necesitas el mismo gráfico pero separado por grupos. FacetGrid te lo da: crea un panel para cada valor de una variable categórica. Ideal si dices "quiero ver el scatter de edad vs tarifa, pero separado por clase".

In [ ]:
g = sns.FacetGrid(
    titanic,
    col="pclass",
    hue="survived",
    height=3.5,
    aspect=1
)

g.map_dataframe(
    sns.histplot,
    x="age",
    kde=True,
    bins=15,
    stat="density"
)

g.add_legend(title="Survived")
g.fig.suptitle("Distribución de edades por clase del Titanic", y=1.01)
plt.tight_layout()

Ves tres paneles: una por cada clase. Dentro de cada panel: rojo (murió) vs azul (sobrevivió). De una mirada ves que la clase 1 tenía pasajeros mayores y una tasa de supervivencia mucho mejor. Eso es FacetGrid.

### Ejercicio 2

Usa el dataset `tips` (mesadas). Crea un FacetGrid con una columna por día (`day`) y colorea por sexo (`sex`). Gráfico: scatter de `total_bill` vs `tip`.

In [ ]:
tips = sns.load_dataset("tips")

g = sns.FacetGrid(
    tips,
    col="___",
    hue="___",
    height=3,
    col_order=["Thurs", "Fri", "Sat", "Sun"]
)

g.map_dataframe(sns.scatterplot, x="total_bill", y="tip", s=80, alpha=0.6)
g.add_legend(title="Sex")
g.fig.suptitle("Propina vs Total por Día de la Semana", y=1.01)
plt.tight_layout()

In [ ]:
assert "g" in dir(), "El objeto FacetGrid no existe"
assert g.fig is not None
print("✅ Ejercicio 2 completado")

In [ ]:
# %% [solution]
# g = sns.FacetGrid(
#     tips,
#     col="day",
#     hue="sex",
#     height=3,
#     col_order=["Thurs", "Fri", "Sat", "Sun"]
# )
# g.map_dataframe(sns.scatterplot, x="total_bill", y="tip", s=80, alpha=0.6)
# g.add_legend(title="Sex")
# g.fig.suptitle("Propina vs Total por Día de la Semana", y=1.01)
# plt.tight_layout()

---

## 3. Plotly Express · gráficos que la gente toca

Matplotlib y Seaborn te dan gráficos estáticos. Están bien para papers y reportes PDF. Pero si necesitas que alguien en una reunión haga zoom, vea los valores exactos con hover, o filtre clicando en la leyenda, necesitas Plotly.

Lo mejor es que la sintaxis es casi idéntica a Seaborn. Una línea. DataFrame adentro, gráfico afuera.

In [ ]:
fig = px.scatter(
    titanic,
    x="age",
    y="fare",
    color="survived",
    size="pclass",
    hover_data=["name", "sex", "pclass"],
    color_discrete_map={0: "#d62728", 1: "#2ca02c"},
    title="Titanic: Edad vs Tarifa (interactivo)",
    labels={"age": "Edad (años)", "fare": "Tarifa (£)", "survived": "Sobrevivió"}
)

fig.update_layout(height=500, font=dict(size=12))
fig.show()

Probá hacer zoom (arrastrá un rectángulo), pasá el mouse sobre los puntos (ves el nombre), y hacé click en la leyenda para mostrar/ocultar categorías. Eso es Plotly.

### Ejercicio 3

Crea un gráfico de barras con Plotly que muestre **% de supervivencia por clase y sexo**. Usa `barmode="group"` para que aparezcan lado a lado (no apiladas).

Pasos: agrupa por clase y sexo, calcula el promedio de `survived`, resetea el índice, y usa `px.bar()`.

In [ ]:
agg_superviv = titanic.groupby(["pclass", "sex"])["survived"].mean().reset_index()
agg_superviv.columns = ["pclass", "sex", "survival_rate"]

fig_bar = px.___(  # ← completa
    agg_superviv,
    x="___",
    y="___",
    color="___",
    barmode="___",
    title="Tasa de supervivencia por clase y sexo",
    labels={"survival_rate": "Tasa de supervivencia", "pclass": "Clase", "sex": "Sexo"}
)

fig_bar.update_layout(height=500)
fig_bar.show()

In [ ]:
assert "fig_bar" in dir()
assert agg_superviv.shape[0] > 0
print("✅ Ejercicio 3 completado")

In [ ]:
# %% [solution]
# fig_bar = px.bar(
#     agg_superviv,
#     x="pclass",
#     y="survival_rate",
#     color="sex",
#     barmode="group",
#     title="Tasa de supervivencia por clase y sexo",
#     labels={"survival_rate": "Tasa de supervivencia", "pclass": "Clase", "sex": "Sexo"}
# )
# fig_bar.update_layout(height=500)
# fig_bar.show()

---

## 4. Heatmap de correlación

Cuando tienes muchas variables numéricas, una matriz de correlación te dice de un vistazo qué pares se relacionan. Los valores positivos (rojo) = al subir una, sube la otra. Negativos (azul) = relación inversa. Cero (blanco) = sin relación.

In [ ]:
numericas = titanic[["age", "fare", "pclass", "survived"]].dropna()
corr_matrix = numericas.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    cbar_kws={"label": "Correlación"}
)
plt.title("Matriz de correlación · Titanic", fontsize=12, fontweight="bold")
plt.tight_layout()

---

## Desafío · Dashboard exploratorio

Necesito que construyas un análisis visual **completo** del dataset `diamonds`. Incluye:

1. Un `pairplot` con al menos 4 variables numéricas, coloreado por `cut`.
2. Un `heatmap de correlación` de las variables numéricas.
3. Un gráfico Plotly (scatter, box o violin) que muestre cómo el precio (`price`) varía según calidad de corte, color o claridad.

Cada gráfico debe tener título claro y labels en ejes. Después, escribe 3-4 oraciones con hallazgos debajo de cada uno.

In [ ]:
diamonds = sns.load_dataset("diamonds")
print(f"Shape: {diamonds.shape}")
print(f"Columnas: {diamonds.columns.tolist()}")
diamonds.head()

Escribí tu código acá para el **pairplot:**

In [ ]:
# ← Tu código aquí

**Hallazgos del pairplot:**

← Escribe aquí 2-3 oraciones explicando qué ves.

Ahora el **heatmap:**

In [ ]:
# ← Tu código aquí

**Hallazgos:**

← ¿Cuál es la correlación más fuerte? ¿Alguna sorpresa?

Y el **Plotly:**

In [ ]:
# ← Tu código aquí (elige scatter, box, violin, etc.)

**Hallazgos:**

← ¿Qué aprendiste? ¿Outliers? ¿Grupos claros?

In [ ]:
assert diamonds.shape[0] > 0, "Los datos no se cargaron"
print("✅ Desafío entregado (revisión manual recomendada)")

---

## Síntesis de lo que vimos

Pairplot para exploración rápida de muchas variables. FacetGrid para comparar patrones entre grupos. Heatmap para ver correlaciones de un vistazo. Plotly cuando alguien en una reunión quiere clickear los datos.

La regla de oro: cada gráfico responde una pregunta clara. Si tu gráfico necesita explicación larga, probablemente estés mostrando demasiado.

Mañana vemos tests estadísticos. La visualización te muestra el patrón. Los tests te dicen si es real o solo ruido.

> **Recordatorio** · Notebook ejecutado al repo antes de la próxima sesión.

---

## Trabajo en equipo

Hoy en el proyecto:

- Crea un pairplot o FacetGrid de tu dataset. Guárdalo.
- Genera la matriz de correlación e identifica las 3 pares más correlacionadas.
- Diseña 1 gráfico Plotly que pueda ir al dashboard final (ese que presentarás al final del mes).

Entrega: un notebook `eda_visualizations.ipynb` con los 3 gráficos + interpretación escrita (5-7 oraciones por cada uno).